In [6]:
import os
import time
from logging import INFO
import pandas as pd

from pathlib import Path
from pollen_worker.virtual_client import VirtualClient, gen_client_fn
from flwr.common import NDArrays, log
from flwr.server.strategy.aggregate import aggregate, aggregate_median

# srun -w mauao -c 24 --gres=gpu:3 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
HOME_DIR = Path(f"{os.getenv('HOME', '')}")

In [2]:
task = "shakespeare_memory"
client_fn = gen_client_fn(task)
n_clients = 100
clients: list[VirtualClient] = [client_fn(0) for _ in range(n_clients)]
fake_fit_res: list[tuple[NDArrays, int]] = [(client.get_parameters({}), 1) for client in clients]
start_time = time.time()
aggregate(fake_fit_res)
log(INFO, f"Time elapsed: {time.time() - start_time}")

INFO flwr 2024-01-22 19:53:08,603 | 1305355685.py:8 | Time elapsed: 0.15612101554870605


In [11]:
if not (HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"fedavg_aggregation_scaling_results.txt").exists():
    results_dict: dict[int, dict[str, float]] = {}
    for n_clients in [10, 100, 1000]:
        results_dict[n_clients]: dict[str, float] = {}
        for task in ["openimage", "google_speech", "reddit", "shakespeare_memory"]:
            client_fn = gen_client_fn(task)
            clients: list[VirtualClient] = [client_fn(0) for _ in range(n_clients)]
            fake_fit_res: list[tuple[NDArrays, int]] = [(client.get_parameters({}), 1) for client in clients]
            start_time = time.time()
            aggregate(fake_fit_res)
            elapsed_time = time.time() - start_time
            results_dict[n_clients][task] = elapsed_time
            log(INFO, f"Aggregating {n_clients} {task} clients, time elapsed: {elapsed_time}")
    with open(HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"fedavg_aggregation_scaling_results.txt", "w") as f:
        f.write(str(results_dict))
else:
    log(INFO, "Weighted aggregation scaling results already exist, skipping")
    # Load results
    with open(HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"fedavg_aggregation_scaling_results.txt", "r") as f:
        results_dict = eval(f.read())
    df = pd.DataFrame(results_dict).transpose()
    print(df.to_latex())

INFO flwr 2024-01-22 19:55:14,668 | 2031267246.py:17 | Weighted aggregation scaling results already exist, skipping


\begin{tabular}{lrrrr}
\toprule
 & openimage & google_speech & reddit & shakespeare_memory \\
\midrule
10 & 0.110118 & 1.434586 & 0.708059 & 0.018188 \\
100 & 3.198174 & 6.626195 & 6.609043 & 0.131063 \\
1000 & 27.815226 & 96.099912 & 74.086368 & 2.993237 \\
\bottomrule
\end{tabular}



In [12]:
if not (HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"median_aggregation_scaling_results.txt").exists():
    results_dict: dict[int, dict[str, float]] = {}
    for n_clients in [10, 100, 1000]:
        results_dict[n_clients]: dict[str, float] = {}
        for task in ["openimage", "google_speech", "reddit", "shakespeare_memory"]:
            client_fn = gen_client_fn(task)
            clients: list[VirtualClient] = [client_fn(0) for _ in range(n_clients)]
            fake_fit_res: list[tuple[NDArrays, int]] = [(client.get_parameters({}), 1) for client in clients]
            start_time = time.time()
            aggregate_median(fake_fit_res)
            elapsed_time = time.time() - start_time
            results_dict[n_clients][task] = elapsed_time
            log(INFO, f"Aggregating {n_clients} {task} clients, time elapsed: {elapsed_time}")
    with open(HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"median_aggregation_scaling_results.txt", "w") as f:
        f.write(str(results_dict))
else:
    log(INFO, "Median aggregation scaling results already exist, skipping")
    # Load results
    with open(HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"median_aggregation_scaling_results.txt", "r") as f:
        results_dict = eval(f.read())
    df = pd.DataFrame(results_dict).transpose()
    print(df.to_latex())


INFO flwr 2024-01-22 19:55:16,969 | 3981874227.py:17 | Median aggregation scaling results already exist, skipping


\begin{tabular}{lrrrr}
\toprule
 & openimage & google_speech & reddit & shakespeare_memory \\
\midrule
10 & 1.446648 & 4.789219 & 1.471002 & 0.178873 \\
100 & 12.762221 & 46.415226 & 18.587834 & 1.582316 \\
1000 & 167.664412 & 812.011429 & 352.457612 & 30.475524 \\
\bottomrule
\end{tabular}

